# Data Split: Companion Notebook

> Companion for `src/my_mlops_project/pipelines/data_split/`.
> Shows the stratified train/val/test split, the preserved class balance, and
> the drift baseline.

**Purpose:** carve reproducible, stratified splits and snapshot the training
distribution as the **drift baseline** for week-6 monitoring.

**Inputs:** `data/04_feature/feature_table.parquet`.
**Outputs:** `X_train/X_val/X_test`, `y_train/y_val/y_test` (05_model_input),
`reference_distribution` (03_primary).

## Table of Contents
1. [Setup](#1-setup)
2. [Load features](#2-load-features)
3. [Split (stratified, seeded)](#3-split-stratified-seeded)
4. [Did stratification preserve the class balance?](#4-did-stratification-preserve-the-class-balance)
5. [The drift baseline](#5-the-drift-baseline)
6. [Validate](#6-validate)
7. [Notes for the report](#7-notes-for-the-report)

## 1. Setup

In [1]:
import sys
from pathlib import Path
import yaml
import pandas as pd

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 180)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))
DATA = PROJECT_ROOT / "data"

params = yaml.safe_load(open(PROJECT_ROOT / "conf" / "base" / "parameters.yml"))["data_split"]
params

{'test_size': 0.2,
 'val_size': 0.1,
 'stratify': True,
 'shuffle': True,
 'random_seed': 42,
 'target_column': 'default',
 'id_columns': ['customer_id', 'event_time']}

## 2. Load features

In [2]:
feature_table = pd.read_parquet(DATA / "04_feature" / "feature_table.parquet")
print("feature_table:", feature_table.shape)
print((feature_table.head()).to_string())

feature_table: (29965, 36)
   customer_id  LIMIT_BAL  SEX  EDUCATION  MARRIAGE  AGE  PAY_0  PAY_2  PAY_3  PAY_4  PAY_5  PAY_6  BILL_AMT1  BILL_AMT2  BILL_AMT3  BILL_AMT4  BILL_AMT5  BILL_AMT6  PAY_AMT1  PAY_AMT2  PAY_AMT3  PAY_AMT4  PAY_AMT5  PAY_AMT6  default  pay_delay_max  pay_delay_mean  n_months_delayed      avg_bill  max_bill  bill_trend  avg_pay_amt  total_pay  util_ratio  pct_paid event_time
0            0      20000    2          2         1   24      2      2     -1     -1     -2     -2       3913       3102        689          0          0          0         0       689         0         0         0         0        1              2       -0.333333                 2   1284.000000      3913        3913   114.833333        689    0.064200  0.089434 2024-01-01
1            1     120000    2          2         2   26     -1      2      0      0      0      2       2682       1725       2682       3272       3455       3261         0      1000      1000      1000         0      2

## 3. Split (stratified, seeded)

We call the production node. The split is **stratified** on the target (keeps the
default rate equal across splits) and **seeded** (`random_seed=42`) so it
reproduces exactly. The target and the feature-store id/time columns are removed
from the feature matrix.

In [3]:
from my_mlops_project.pipelines.data_split.nodes import split_data, make_reference_distribution

X_train, X_val, X_test, y_train, y_val, y_test = split_data(feature_table, params)
print("feature matrix columns:", X_train.shape[1], "(target + customer_id + event_time removed)")

split_data: train=(20975, 33) val=(2997, 33) test=(5993, 33) | default rate train/val/test = 0.221/0.221/0.221
feature matrix columns: 33 (target + customer_id + event_time removed)


## 4. Did stratification preserve the class balance?

The whole point of stratifying: every split should carry the same ~22% default
rate as the full data. If train/val/test diverged, the model would train and be
evaluated on different populations.

In [4]:
balance = pd.DataFrame({
    "rows": [len(y_train), len(y_val), len(y_test)],
    "default_rate": [float(y_train["default"].mean()),
                     float(y_val["default"].mean()),
                     float(y_test["default"].mean())],
}, index=["train", "val", "test"])
balance["pct_of_total"] = (balance["rows"] / balance["rows"].sum() * 100).round(1)
print("overall default rate:", round(feature_table["default"].mean(), 3))
print((balance.round(3)).to_string())

overall default rate: 0.221
        rows  default_rate  pct_of_total
train  20975         0.221          70.0
val     2997         0.221          10.0
test    5993         0.221          20.0


## 5. The drift baseline

`reference_distribution` is a snapshot of the **training features**. In week-6
monitoring, `data_drifts` compares each production batch against this baseline
(PSI/JS per feature, PCA reconstruction error). Fixing it to the training data is
what makes "has the world moved away from what we trained on?" answerable.

In [5]:
reference = make_reference_distribution(X_train)
print("drift baseline:", reference.shape)
print((reference.describe().T[["mean", "std", "min", "max"]].head(8).round(2)).to_string())

make_reference_distribution: baseline snapshot (20975, 33)
drift baseline: (20975, 33)
                mean        std      min        max
LIMIT_BAL  167012.62  129768.45  10000.0  1000000.0
SEX             1.60       0.49      1.0        2.0
EDUCATION       1.84       0.74      1.0        4.0
MARRIAGE        1.56       0.52      1.0        3.0
AGE            35.47       9.21     21.0       75.0
PAY_0          -0.01       1.12     -2.0        8.0
PAY_2          -0.13       1.20     -2.0        7.0
PAY_3          -0.16       1.19     -2.0        8.0


## 6. Validate

In [6]:
assert len(X_train) + len(X_val) + len(X_test) == len(feature_table), "splits must sum to the whole"
for bad in ("default", "customer_id", "event_time"):
    assert bad not in X_train.columns, f"{bad} leaked into features"
assert abs(y_train["default"].mean() - feature_table["default"].mean()) < 0.02, "train class balance drifted"
print("All split assertions passed.")

All split assertions passed.


## 7. Notes for the report

> For [`../report/REPORT_OUTLINE.md`](../report/REPORT_OUTLINE.md) section 2 (planning) and section 3 (modelling setup).

- **Stratified, seeded split**, 70/10/20 train/val/test, default rate held at
  ~22% in every split (table in section 4). The fixed seed (`random_seed=42`) makes the
  partition reproducible, satisfying "everyone produces the same results".
- **No leakage**. Target and feature-store id/time columns are excluded from the
  feature matrix.
- **Drift baseline**. The training distribution is snapshotted as
  `reference_distribution`, the reference the week-6 monitoring compares against.